In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import json
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import os
import math

In [3]:
#where all the host folders and the jsons live
main_folder = '.'
host_folders = [folder for folder in os.listdir(main_folder) if os.path.isdir(os.path.join(main_folder, folder))]

#what we are plotting
all_genes = []
all_hosts = []
all_slopes = [] #rate of adaptation
all_lower_95cis = []
all_upper_95cis = []

for host_folder in host_folders:
    host_path = os.path.join(main_folder, host_folder)
    json_files = [file for file in os.listdir(host_path) if file.endswith('.json')]
    for json_file in json_files:
        json_path = os.path.join(host_path, json_file)

        with open(json_path, 'r') as file:
            data = json.load(file)

        rate_of_adaptation = data["rate_of_adaptation"]
        bootstrap_rate_of_adaptation = data["bootstrap_rate_of_adaptation"]

        slope_sci = rate_of_adaptation * (10**3)
        bs_slope_sci = [x * (10**3) for x in bootstrap_rate_of_adaptation]
        lower_95ci = np.percentile(sorted(bs_slope_sci), 2.5)
        upper_95ci = np.percentile(sorted(bs_slope_sci), 97.5)

        all_genes.append(data['gene'].upper()) 
        all_hosts.append(data['host'].replace(' ', '\n')) 
        all_slopes.append(slope_sci)
        all_lower_95cis.append(lower_95ci)
        all_upper_95cis.append(upper_95ci)


#transforming data into df
all_data = {
    'gene': all_genes,
    'host': all_hosts,
    'adaptive_subs_per_codon_per_year': all_slopes,
    'lower_95ci': all_lower_95cis,
    'upper_95ci': all_upper_95cis,
    'ci': list(zip(all_lower_95cis, all_upper_95cis)),
}

df_all = pd.DataFrame(all_data)


In [4]:
df_all

,gene,host,adaptive_subs_per_codon_per_year,lower_95ci,upper_95ci,ci
0,PB1,swine_NA,0.225863,0.015171,0.431633,"(0.015171215065619026, 0.4316330482938087)"
1,NA,swine_NA,1.820134,0.981262,2.563300,"(0.9812622805235888, 2.5633004110256525)"
2,HA,swine_NA,1.802118,1.004741,2.518504,"(1.0047406331800295, 2.5185041004967785)"
3,N8,eurasian_avian,1.800800,1.295232,2.327783,"(1.2952322476030278, 2.327782975525985)"
4,PB1,eurasian_avian,0.024611,-0.000090,0.105950,"(-9.0168540154121e-05, 0.105949749494244)"
5,HA,eurasian_avian,0.226052,0.037753,0.433110,"(0.037752754226104725, 0.43311039915871075)"
6,N2,eurasian_avian,2.057096,0.945532,3.177144,"(0.9455323768372771, 3.177143848657034)"
7,PB1,equine,0.387234,0.093363,0.687866,"(0.09336259980672833, 0.6878661168618216)"
8,NA,equine,1.960826,1.496995,2.679543,"(1.4969950820090026, 2.679543165082331)"
9,HA,equine,1.407715,0.948009,1.882073,"(0.9480087554226315, 1.8820732497065902)"


In [ ]:
color_and_host_order_map = {
    'eurasian_avian': "#fb6a4a",
    'na_avian': "#b6282e",
    'swine_euro': "#133359",
    'swine_NA': '#33659e',
    'canine_h3n2': '#7fbadc',
    'human': '#f4a261',
    'equine': '#e9c46a'
}

genes = list(map(str.upper, ['HA', 'NA', 'PB1']))
hosts = list(map(str.upper, [
    'eurasian_avian',
    'na_avian',
    'swine_euro',
    'swine_NA',
    'canine_h3n2',
    'human',
    'equine'
]))

df_to_use = df_all.copy()
df_to_use['color'] = df_to_use['host'].map(color_and_host_order_map)

number_of_cols = 3
number_of_plots = len(genes)
number_of_rows = math.ceil(number_of_plots/number_of_cols)

fig = make_subplots(
    subplot_titles=genes,
    rows=number_of_rows, 
    cols=number_of_cols,
    shared_yaxes='all',
    vertical_spacing=0.40,
    horizontal_spacing = 0.05
)

for idx, gene in enumerate(genes):
    # Handle NA gene specially to include N2 and N8
    if gene == 'NA':
        subset_df = df_to_use[df_to_use["gene"].isin(['NA', 'N2', 'N8'])]
    else:
        subset_df = df_to_use[df_to_use["gene"] == gene]
    
    row_idx = idx // number_of_cols + 1
    column_idx = idx % number_of_cols + 1
    
    for host_idx, host in enumerate(hosts):
        sub_subset_df = subset_df[subset_df["host"].str.upper() == host]

        if not sub_subset_df.empty:
            # Create x-offset based on gene type for NA subplot
            if gene == 'NA':
                gene_offset_map = {'N2': -0.25, 'NA': 0, 'N8': 0.25}  # Increased spacing
                x_offset = sub_subset_df["gene"].map(gene_offset_map)
                x_values = [host_idx + offset for offset in x_offset]
            else:
                x_values = [host_idx] * len(sub_subset_df)
            
            fig.add_trace(
                go.Scatter(
                    x=x_values, 
                    y=sub_subset_df["adaptive_subs_per_codon_per_year"],
                    name=host,
                    text=sub_subset_df["gene"],
                    marker=dict(
                        color=sub_subset_df["color"].values[0],
                        size=10
                    ),
                    mode = "markers",
                    error_y=dict(
                        type = 'data',
                        array = sub_subset_df["upper_95ci"] - sub_subset_df["adaptive_subs_per_codon_per_year"],
                        arrayminus = sub_subset_df["adaptive_subs_per_codon_per_year"] - sub_subset_df["lower_95ci"],
                        visible = True,
                        color = sub_subset_df["color"].values[0],
                        width=0,
                        thickness=3
                    ),
                    legendgroup=host,
                    showlegend= idx == 1
                ),
                row = row_idx, 
                col = column_idx
            )
        
    getattr(fig.layout, 'yaxis' if idx == 0 else f'yaxis{idx}').showticklabels = True

fig.update_annotations(font_size=25)
            
fig.update_layout(
    font_size=18,
    width=1600,  
    height=500
)

# set x-axis labels to show host names
hosts_lower = [h.lower() for h in hosts]
for i in range(number_of_plots):
    xaxis_name = 'xaxis' if i == 0 else f'xaxis{i+1}'
    getattr(fig.layout, xaxis_name).tickvals = list(range(len(hosts)))
    getattr(fig.layout, xaxis_name).ticktext = hosts_lower

fig.update_xaxes(tickangle=-55, tickfont_size=20)
fig.update_yaxes(tickfont_size=20)
fig.update_layout(showlegend=False)
    
fig.show()



/Users/monclalab1/anaconda3/lib/python3.10/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.


